# Matching System Prototype con preferenze di razza

Questo notebook implementa un prototipo funzionante del sistema di matching cane–famiglia.

Rispetto alla versione precedente, viene aggiunta anche la **preferenza sulle razze** espresse dalla famiglia.

Il sistema utilizza:

- caratteristiche strutturate dei cani;
- preferenze espresse dalla famiglia;
- razze preferite dalla famiglia;
- descrizione testuale del cane ideale;
- embedding BERT delle descrizioni dei cani;

per generare una classifica dei cani più compatibili con un profilo adottante.


## 1. Import delle librerie

In [1]:
import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

## 2. Caricamento dei dati

Vengono caricati:

- il dataset pulito dei cani (`dogs_clean.csv`);
- gli embedding BERT delle descrizioni degli annunci (`bert_embeddings.npy`).

Se il dataset non contiene ancora `breed1_label` e `breed2_label`, queste colonne vengono create automaticamente usando `breed_labels.csv`.


In [2]:
dogs = pd.read_csv("../data/processed/dogs_clean.csv")
bert_embeddings = np.load("../data/processed/bert_embeddings.npy")

# Se le colonne testuali delle razze non sono presenti,
# vengono create usando il file di mapping breed_labels.csv.
if "breed1_label" not in dogs.columns or "breed2_label" not in dogs.columns:
    breed_labels = pd.read_csv("../data/raw/breed_labels.csv")

    breed_map = dict(
        zip(
            breed_labels["BreedID"],
            breed_labels["BreedName"]
        )
    )

    dogs["breed1_label"] = dogs["Breed1"].map(breed_map)
    dogs["breed2_label"] = dogs["Breed2"].map(breed_map)
    dogs["breed2_label"] = dogs["breed2_label"].fillna("None")

print("Dataset cani:", dogs.shape)
print("Embedding BERT:", bert_embeddings.shape)

dogs[[
    "Name",
    "breed1_label",
    "breed2_label",
    "age_group",
    "gender_label",
    "maturity_size_label",
    "fur_length_label"
]].head()

Dataset cani: (8132, 32)
Embedding BERT: (8132, 768)


,Name,breed1_label,breed2_label,age_group,gender_label,maturity_size_label,fur_length_label
0,Brisco,Mixed Breed,NaN,puppy,male,medium,medium
1,Miko,Mixed Breed,NaN,puppy,female,medium,short
2,Hunter,Mixed Breed,NaN,puppy,male,medium,short
3,Siu Pak & Her 6 Puppies,Mixed Breed,NaN,puppy,female,medium,short
4,Bear,Mixed Breed,NaN,puppy,male,medium,short


## 3. Profilo famiglia adottante

In questa sezione viene definito un esempio di profilo famiglia.

Il campo `preferred_breeds` contiene una o più razze desiderate.  
Se la lista è vuota, significa che la famiglia non ha preferenze di razza.

Esempi:

```python
"preferred_breeds": ["Labrador Retriever", "Golden Retriever"]
```

oppure:

```python
"preferred_breeds": []
```


In [3]:
family_profile = {
    "applicant_age": "31-60",              # 18-30, 31-60, over_60
    "children": True,
    "dog_experience": "some",              # none, some, multiple, high
    "housing": "house",                    # apartment, house, countryside
    "garden": True,

    "preferred_gender": "indifferent",     # male, female, indifferent
    "preferred_age": "puppy",              # puppy, young, adult, senior, indifferent
    "preferred_size": "medium",            # small, medium, large, extra_large, indifferent
    "preferred_fur": "short",              # short, medium, long, indifferent

    # Nuovo campo: razze preferite
    "preferred_breeds": [
        "Labrador Retriever",
        "Golden Retriever",
        "Mixed Breed"
    ],

    "daily_time": "2-4",                   # <1, 1-2, 2-4, >4
    "activity_level": "moderate",          # calm, moderate, active

    "ideal_dog_description": (
        "We are looking for a friendly and affectionate dog, suitable for a family with children, "
        "not too large, playful but also calm at home."
    )
}

family_profile

{'applicant_age': '31-60',
 'children': True,
 'dog_experience': 'some',
 'housing': 'house',
 'garden': True,
 'preferred_gender': 'indifferent',
 'preferred_age': 'puppy',
 'preferred_size': 'medium',
 'preferred_fur': 'short',
 'preferred_breeds': ['Labrador Retriever', 'Golden Retriever', 'Mixed Breed'],
 'daily_time': '2-4',
 'activity_level': 'moderate',
 'ideal_dog_description': 'We are looking for a friendly and affectionate dog, suitable for a family with children, not too large, playful but also calm at home.'}

## 4. Regole di compatibilità strutturata

La compatibilità strutturata confronta le preferenze della famiglia con le caratteristiche del cane.

In questa versione vengono considerati anche `breed1_label` e `breed2_label`.

La razza ha un peso elevato:

- se la razza principale (`breed1_label`) è tra quelle richieste → punteggio massimo;
- se la seconda razza (`breed2_label`) è tra quelle richieste → punteggio parziale;
- se nessuna razza corrisponde → punteggio basso;
- se la famiglia non indica razze preferite → nessuna penalizzazione.

Il sistema distingue inoltre tra:

- vincoli hard;
- vincoli soft;
- preferenze pesate.


In [4]:
def normalize_text(text):
    return str(text).strip().lower()


def match_preference(preference, dog_value):
    if preference == "indifferent":
        return 1.0
    return 1.0 if preference == dog_value else 0.0


def size_similarity(preferred_size, dog_size):
    if preferred_size == "indifferent":
        return 1.0

    size_order = {
        "small": 0,
        "medium": 1,
        "large": 2,
        "extra_large": 3
    }

    if preferred_size not in size_order or dog_size not in size_order:
        return 0.0

    distance = abs(size_order[preferred_size] - size_order[dog_size])

    if distance == 0:
        return 1.0
    elif distance == 1:
        return 0.5
    else:
        return 0.0


def compute_breed_score(dog, family):
    preferred_breeds = family.get("preferred_breeds", [])

    # Nessuna preferenza di razza: non penalizza
    if len(preferred_breeds) == 0:
        return 1.0

    preferred_breeds = [
        normalize_text(breed)
        for breed in preferred_breeds
    ]

    breed1 = normalize_text(dog.get("breed1_label", ""))
    breed2 = normalize_text(dog.get("breed2_label", ""))

    # Razza principale corrispondente: massimo punteggio
    if breed1 in preferred_breeds:
        return 1.0

    # Seconda razza corrispondente: punteggio parziale
    if breed2 in preferred_breeds:
        return 0.6

    # Nessuna corrispondenza
    return 0.0

## 5. Funzione di compatibilità strutturata con peso sulla razza

Invece di una media semplice, viene usata una **media pesata**.

La razza ha peso 3, quindi influenza maggiormente lo score rispetto ad altre preferenze come sesso o pelo.


In [5]:
def compute_structured_score(dog, family):
    # Hard constraint: meno di 1 ora al giorno non è compatibile con l'adozione.
    if family["daily_time"] == "<1":
        return 0.0

    # Hard constraint: cani extra large richiedono un giardino.
    if dog["maturity_size_label"] == "extra_large" and not family["garden"]:
        return 0.0

    # Hard constraint: nessuna esperienza + cane extra large.
    if family["dog_experience"] == "none" and dog["maturity_size_label"] == "extra_large":
        return 0.0

    weighted_scores = []

    # Preferenze dirette
    gender_score = match_preference(family["preferred_gender"], dog["gender_label"])
    age_score = match_preference(family["preferred_age"], dog["age_group"])
    size_score = size_similarity(family["preferred_size"], dog["maturity_size_label"])
    fur_score = match_preference(family["preferred_fur"], dog["fur_length_label"])

    weighted_scores.append((gender_score, 1))
    weighted_scores.append((age_score, 1))
    weighted_scores.append((size_score, 1))
    weighted_scores.append((fur_score, 1))

    # Razza: peso alto
    breed_score = compute_breed_score(dog, family)
    weighted_scores.append((breed_score, 3))

    # Regola bambini
    if family["children"]:
        children_score = 1.0 if dog["age_group"] in ["puppy", "young"] else 0.5
        weighted_scores.append((children_score, 1))

    # Regola giardino e taglia
    if dog["maturity_size_label"] == "large":
        garden_score = 1.0 if family["garden"] else 0.2
        weighted_scores.append((garden_score, 1))
    elif dog["maturity_size_label"] == "extra_large":
        weighted_scores.append((1.0, 1))
    else:
        weighted_scores.append((1.0, 1))

    # Regola tempo disponibile
    if family["daily_time"] == "1-2":
        time_score = 0.6 if dog["age_group"] == "puppy" else 0.8
        weighted_scores.append((time_score, 1))
    else:
        weighted_scores.append((1.0, 1))

    # Regola esperienza
    if family["dog_experience"] == "none":
        if dog["maturity_size_label"] == "large":
            experience_score = 0.2
        elif dog["age_group"] == "puppy":
            experience_score = 0.5
        else:
            experience_score = 1.0
    else:
        experience_score = 1.0

    weighted_scores.append((experience_score, 1))

    # Regola richiedente over 60
    if family["applicant_age"] == "over_60":
        senior_score = 1.0 if dog["age_group"] in ["adult", "senior"] else 0.4
        weighted_scores.append((senior_score, 1))

    total_score = sum(score * weight for score, weight in weighted_scores)
    total_weight = sum(weight for score, weight in weighted_scores)

    return total_score / total_weight

## 6. Calcolo della compatibilità strutturata

La funzione viene applicata a tutti i cani del dataset.

Vengono aggiunte due colonne:

- `breed_score`: punteggio specifico sulla razza;
- `structured_score`: punteggio strutturato complessivo.


In [6]:
dogs["breed_score"] = dogs.apply(
    lambda row: compute_breed_score(row, family_profile),
    axis=1
)

dogs["structured_score"] = dogs.apply(
    lambda row: compute_structured_score(row, family_profile),
    axis=1
)

dogs[[
    "Name",
    "breed1_label",
    "breed2_label",
    "breed_score",
    "age_group",
    "gender_label",
    "maturity_size_label",
    "fur_length_label",
    "structured_score"
]].head()

,Name,breed1_label,breed2_label,breed_score,age_group,gender_label,maturity_size_label,fur_length_label,structured_score
0,Brisco,Mixed Breed,NaN,1.0,puppy,male,medium,medium,0.909091
1,Miko,Mixed Breed,NaN,1.0,puppy,female,medium,short,1.000000
2,Hunter,Mixed Breed,NaN,1.0,puppy,male,medium,short,1.000000
3,Siu Pak & Her 6 Puppies,Mixed Breed,NaN,1.0,puppy,female,medium,short,1.000000
4,Bear,Mixed Breed,NaN,1.0,puppy,male,medium,short,1.000000


## 7. Similarità semantica tramite BERT

La descrizione del cane ideale inserita dalla famiglia viene trasformata in embedding BERT.

Successivamente viene calcolata la similarità coseno tra:

- embedding della descrizione della famiglia;
- embedding della descrizione di ogni cane.


In [7]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)
bert_model.eval()

print("BERT model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model loaded.


In [8]:
def get_bert_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = bert_model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.squeeze().numpy()


family_embedding = get_bert_embedding(family_profile["ideal_dog_description"])

semantic_scores = cosine_similarity(
    family_embedding.reshape(1, -1),
    bert_embeddings
).flatten()

dogs["bert_similarity"] = (semantic_scores + 1) / 2

dogs[["Name", "bert_similarity"]].head()

,Name,bert_similarity
0,Brisco,0.907731
1,Miko,0.885574
2,Hunter,0.913940
3,Siu Pak & Her 6 Puppies,0.832422
4,Bear,0.837586


## 8. Score finale di compatibilità

Lo score finale combina:

- 70% compatibilità strutturata;
- 30% similarità semantica tramite BERT.

La razza influenza lo `structured_score`, quindi entra indirettamente anche nello `final_score`.


In [9]:
dogs["final_score"] = (
    0.7 * dogs["structured_score"] +
    0.3 * dogs["bert_similarity"]
)

dogs[[
    "Name",
    "breed1_label",
    "breed2_label",
    "breed_score",
    "structured_score",
    "bert_similarity",
    "final_score"
]].head()

,Name,breed1_label,breed2_label,breed_score,structured_score,bert_similarity,final_score
0,Brisco,Mixed Breed,NaN,1.0,0.909091,0.907731,0.908683
1,Miko,Mixed Breed,NaN,1.0,1.000000,0.885574,0.965672
2,Hunter,Mixed Breed,NaN,1.0,1.000000,0.913940,0.974182
3,Siu Pak & Her 6 Puppies,Mixed Breed,NaN,1.0,1.000000,0.832422,0.949727
4,Bear,Mixed Breed,NaN,1.0,1.000000,0.837586,0.951276


## 9. Classifica finale dei cani consigliati

I cani vengono ordinati in base allo score finale.

La classifica tiene conto di:

- preferenze strutturate;
- preferenze di razza;
- similarità semantica tramite BERT;
- vincoli hard e soft.


In [10]:
ranking = dogs.sort_values(
    by="final_score",
    ascending=False
)[[
    "Name",
    "Age",
    "breed1_label",
    "breed2_label",
    "breed_score",
    "age_group",
    "gender_label",
    "maturity_size_label",
    "fur_length_label",
    "PhotoAmt",
    "structured_score",
    "bert_similarity",
    "final_score",
    "Description"
]].head(10)

ranking

,Name,Age,breed1_label,breed2_label,breed_score,age_group,gender_label,maturity_size_label,fur_length_label,PhotoAmt,structured_score,bert_similarity,final_score,Description
2433,Leo,3,Mixed Breed,NaN,1.0,puppy,male,medium,short,3.0,1.0,0.943833,0.983150,Leo is a stray awaiting a nice place to be cal...
5902,Mick,3,Mixed Breed,NaN,1.0,puppy,male,medium,short,5.0,1.0,0.943298,0.982989,Mick is a very handsome mixed breed pup. A bit...
5663,Chino,1,Mixed Breed,NaN,1.0,puppy,male,medium,short,11.0,1.0,0.940831,0.982249,Chino is among 7 siblings that need a good hom...
2119,Gruff,3,Mixed Breed,Terrier,1.0,puppy,male,medium,short,8.0,1.0,0.939150,0.981745,Gruff is a handsome looking black dog. He is p...
2999,Zara,2,Mixed Breed,Doberman Pinscher,1.0,puppy,female,medium,short,13.0,1.0,0.938650,0.981595,Zara is a beautiful young doggie with nice bla...
1566,Mr Zorro,6,Mixed Breed,NaN,1.0,puppy,male,medium,short,11.0,1.0,0.938196,0.981459,"Mr Zorro is an ultra friendly, silly, playful ..."
2231,Mina,3,Mixed Breed,Doberman Pinscher,1.0,puppy,female,medium,short,5.0,1.0,0.937850,0.981355,A mixed breed female dog gave birth to a litte...
5799,"Grey-eyed, Brown Male Puppy",2,Mixed Breed,NaN,1.0,puppy,male,medium,short,2.0,1.0,0.937832,0.981350,This cute grey-eyed boy was found in our taman...
3267,Socks,2,Mixed Breed,American Staffordshire Terrier,1.0,puppy,male,medium,short,3.0,1.0,0.936690,0.981007,ADOPTION FEE COVERS DESEXING AND VACCINES. Soc...
7714,Hugo,5,Mixed Breed,NaN,1.0,puppy,male,medium,short,4.0,1.0,0.936414,0.980924,"Little Hugo is a healthy, bouncy and happy pup..."


## 10. Spiegazione della raccomandazione

Per rendere il sistema più interpretabile, viene generata una breve spiegazione automatica per ciascun cane consigliato.

La spiegazione include anche la corrispondenza di razza, se presente.


In [11]:
def explain_recommendation(dog, family):
    reasons = []

    if family["preferred_age"] == "indifferent" or dog["age_group"] == family["preferred_age"]:
        reasons.append("fascia d'età compatibile")

    if family["preferred_size"] == "indifferent" or dog["maturity_size_label"] == family["preferred_size"]:
        reasons.append("taglia compatibile")

    if family["preferred_gender"] == "indifferent" or dog["gender_label"] == family["preferred_gender"]:
        reasons.append("sesso compatibile")

    if family["preferred_fur"] == "indifferent" or dog["fur_length_label"] == family["preferred_fur"]:
        reasons.append("lunghezza del pelo compatibile")

    if len(family.get("preferred_breeds", [])) > 0:
        if compute_breed_score(dog, family) == 1.0:
            reasons.append("razza principale compatibile")
        elif compute_breed_score(dog, family) == 0.6:
            reasons.append("seconda razza compatibile")

    if dog["bert_similarity"] > dogs["bert_similarity"].quantile(0.75):
        reasons.append("descrizione semanticamente vicina al cane ideale")

    if len(reasons) == 0:
        return "Compatibilità basata sul punteggio complessivo."

    return ", ".join(reasons)


ranking_with_explanations = ranking.copy()

ranking_with_explanations["explanation"] = ranking_with_explanations.apply(
    lambda row: explain_recommendation(row, family_profile),
    axis=1
)

ranking_with_explanations[[
    "Name",
    "breed1_label",
    "breed2_label",
    "final_score",
    "explanation"
]]

,Name,breed1_label,breed2_label,final_score,explanation
2433,Leo,Mixed Breed,NaN,0.983150,"fascia d'età compatibile, taglia compatibile, ..."
5902,Mick,Mixed Breed,NaN,0.982989,"fascia d'età compatibile, taglia compatibile, ..."
5663,Chino,Mixed Breed,NaN,0.982249,"fascia d'età compatibile, taglia compatibile, ..."
2119,Gruff,Mixed Breed,Terrier,0.981745,"fascia d'età compatibile, taglia compatibile, ..."
2999,Zara,Mixed Breed,Doberman Pinscher,0.981595,"fascia d'età compatibile, taglia compatibile, ..."
1566,Mr Zorro,Mixed Breed,NaN,0.981459,"fascia d'età compatibile, taglia compatibile, ..."
2231,Mina,Mixed Breed,Doberman Pinscher,0.981355,"fascia d'età compatibile, taglia compatibile, ..."
5799,"Grey-eyed, Brown Male Puppy",Mixed Breed,NaN,0.981350,"fascia d'età compatibile, taglia compatibile, ..."
3267,Socks,Mixed Breed,American Staffordshire Terrier,0.981007,"fascia d'età compatibile, taglia compatibile, ..."
7714,Hugo,Mixed Breed,NaN,0.980924,"fascia d'età compatibile, taglia compatibile, ..."


## 11. Conclusioni

Il prototipo dimostra come sia possibile integrare anche le preferenze di razza nel sistema di matching cane–famiglia.

La razza viene trattata come una preferenza importante:

- razza principale corrispondente: punteggio massimo;
- seconda razza corrispondente: punteggio parziale;
- nessuna corrispondenza: penalizzazione.

Questo miglioramento rende il sistema più realistico, perché molte famiglie possono avere preferenze esplicite sulla razza del cane.

Il sistema finale combina quindi:

- compatibilità strutturata;
- preferenze di razza;
- similarità semantica tramite BERT;
- spiegazioni automatiche delle raccomandazioni.
